# Charlotte Project | Week 10 — Class 1
## NumPy Fundamentals and Vectorized Sensor Computing

**Format:** Interactive, instructor-led Jupyter Notebook (180 minutes). **Formal submissions:** independent `.py` scripts and GitHub Pull Requests.

**Purpose:** Week 10 develops Python for data analysis. This first class focuses on **NumPy ndarrays and vectorized numeric computing**, the foundation for handling multi-channel energy telemetry. Later classes introduce Pandas, complex indexing and aggregation, time-series processing and cleaning.

**Learning outcomes:** create and inspect arrays; select readings; compute engineering conversions without row-by-row Python loops; filter with masks; handle missing numeric readings; aggregate correctly across an axis.

**Important:** This notebook contains **demonstration data**, different from the graded lab dataset. Modify the examples to learn, but write formal assignments independently in `.py` files.

### Before you begin — readiness check

1. Open your individual Week 10 repository in VS Code.
2. Select its `.venv` interpreter as the notebook kernel.
3. Install `numpy` and `ipykernel` with `python -m pip install numpy ipykernel` in the activated environment.
4. Run the next cell. No external downloads or cloud credentials are needed.

**Notebook etiquette:** Predict outputs before pressing Run, change one variable at a time, and rerun the affected cells in order. The notebook is a teaching playground, not the graded submission.

In [ ]:
import numpy as np
print('NumPy:', np.__version__)
print('Notebook ready for sensor analytics')

# A — What is NumPy, and what is an ndarray? (30 minutes)

### Concept introduction

**NumPy** is a Python library designed for efficient numeric arrays and numerical operations. The `ndarray` is its central data structure: a collection of values arranged into one or more *dimensions*. For sensor monitoring, a **2D array** can hold rows of observations and columns of channels (pressure, temperature, vibration and flow). Arrays have a **shape**, **number of dimensions** (`ndim`), **size** (total cells), and **dtype** (type of stored values).

**Why it matters in oil and gas:** A drilling analyst often receives many simultaneous measurements. A consistent numeric matrix makes it possible to convert a full pressure column, compare thousands of measurements with operating thresholds, and aggregate readings in a compact, auditable way.

### Python lists versus numeric arrays

Python lists are flexible general containers. Multiplying a list by an integer repeats its elements; multiplying a NumPy numeric array multiplies its values *element by element*. NumPy typically uses a common numeric dtype per array, which makes numeric operations predictable. An array is not automatically a replacement for every Python collection.

### Syntax to recognise

- `np.array([...], dtype=float)` creates a numeric array.
- `arr.shape` gives each dimension's length; `(4, 3)` means 4 rows × 3 columns.
- `arr.ndim`, `arr.size`, `arr.dtype` reveal structural details.
- `np.nan` represents a missing numeric reading. It is **not** the same as zero.

**Common mistakes:** treating the first row as index 1 rather than 0; assuming arrays of strings and floats behave like a numeric matrix; forgetting that sensor columns have different units.

**Predict → run → explain:** What will `[2,3]*2` do? What will `np.array([2,3])*2` do?

In [ ]:
ordinary_list = [2, 3]
numeric_array = np.array([2, 3], dtype=float)
print('list * 2:', ordinary_list * 2)
print('NumPy * 2:', numeric_array * 2)

### Demonstration dataset — separate from the graded lab

We have **six synthetic observations** from a teaching example. Each row contains `[pressure_psi, temperature_f, vibration_mm_s, flow_bpd]`. One flow reading is missing. In the real field, we would also keep timestamps, well IDs and provenance alongside these numbers; we omit them here to concentrate on the structure of a numeric matrix.

**Think:** Which axis represents observation time/order? Which axis represents measurement channel?

In [ ]:
import numpy as np
channel_names = ['pressure_psi', 'temperature_f', 'vibration_mm_s', 'flow_bpd']
demo = np.array([
    [3000, 180, 4.0, 420],
    [3450, 190, 5.1, 405],
    [3600, 210, 7.0, np.nan],
    [3250, 200, 4.8, 390],
    [3720, 205, 6.5, 450],
    [2900, 175, 3.2, 410],
], dtype=float)
print('channels:', channel_names)
print('shape:', demo.shape, '| ndim:', demo.ndim, '| size:', demo.size, '| dtype:', demo.dtype)
print(demo)

### A: Quick practice (ungraded)

Without running the code, predict `demo.shape`, `demo.size`, and the number of numeric columns. Then change a demonstration pressure value and rerun. Observe that the matrix remains numeric and its dimensions do not change.

# B — Indexing, slicing and selecting sensor channels (30 minutes)

### Concept introduction

**Indexing** selects one value by position; **slicing** selects ranges. In a 2D array, the notation `matrix[row_index, column_index]` tells NumPy exactly which measurement is needed. The colon `:` means “all positions” along that dimension. A slice like `0:3` includes 0, 1 and 2, **not** 3.

**Engineering use:** Analysts frequently isolate the pressure channel, select a recent inspection window, or inspect a single measurement that appears suspicious.

### Essential patterns

- `demo[0, 0]` first observation's pressure.
- `demo[:, 0]` all pressure readings.
- `demo[1:4, :]` observations at index 1, 2 and 3 across all channels.
- `demo[:, [0, 2]]` only pressure and vibration columns.

**Common mistakes:** `demo[0]` returns a whole row, not the first pressure *column*; Python indexing begins at zero; slicing creates views in many cases, so modifying a slice can change the original array—use `.copy()` when independence matters.

**Think before running:** What will be the shape of `demo[:, [0, 2]]`?

In [ ]:
print('first pressure:', demo[0, 0])
print('pressure column:', demo[:, 0])
print('rows 1 to 3:', demo[1:4, :])
print('pressure and vibration:', demo[:, [0, 2]])

In [ ]:
pressure_psi = demo[:, 0].copy()
print('selected readings:', pressure_psi)
print('first three pressure readings:', pressure_psi[:3])

### B: Try it

Select all temperatures in °F, then select the last two observations. Print the shapes to confirm. **Do not** use a Python loop just to select a column.

In [ ]:
import numpy as np

channel_names = ['pressure_psi', 'temperature_f', 'vibration_mm_s', 'flow_bpd']

demo = np.array([
    [3000, 180, 4.0, 420],
    [3450, 190, 5.1, 405],
    [3600, 210, 7.0, np.nan],
    [3250, 200, 4.8, 390],
    [3720, 205, 6.5, 450],
    [2900, 175, 3.2, 410],
], dtype=float)

# Select all temperatures in Fahrenheit
temperature = demo[:, 1]

# Select the last two observations
last_two = demo[-2:, :]

# Print the results
print('Temperature values:', temperature)
print('Temperature shape:', temperature.shape)

print('Last two observations:')
print(last_two)
print('Last two shape:', last_two.shape)

In [ ]:
# Exploration: edit the column index and slice boundaries.
selected = demo[:, 1]
window = demo[-2:, :]
print('temperatures:', selected)
print('window shape:', window.shape)

# C — Vectorization, arithmetic and broadcasting (30 minutes)

### Concept introduction

**Vectorization** means expressing numeric calculations on entire arrays rather than writing a Python `for` loop over each reading. Arithmetic such as `+`, `-`, `*`, and `/` is applied elementwise to matching shapes. This makes scientific calculations succinct and often more efficient.

**Broadcasting** lets NumPy combine compatible shapes: a scalar can apply to every array element, and a one-dimensional vector with a matching column count can apply across rows of a matrix. Broadcasting is based on array shapes, not on the meanings of units, so you still need engineering judgment.

### Engineering formulas

Pressure: `pressure_kpa = pressure_psi * 6.8947572932`.

Temperature: `temperature_c = (temperature_f - 32) * 5 / 9`.

The result of a conversion should be labeled with its output unit. NaN remains NaN through normal arithmetic, preserving missing data.

### Common mistakes

- Using a whole-matrix multiplier when each channel needs different physics.
- Interpreting `matrix * [factor_1, ...]` as all columns having the same physical quantity.
- Confusing `*` (elementwise multiplication) with matrix multiplication (`@`).

**Prediction:** Can one numeric pressure column be converted using a single multiplication expression? Yes. What happens to missing values?

In [ ]:
pressure_kpa = demo[:, 0] * 6.8947572932
temperature_c = (demo[:, 1] - 32) * 5 / 9
print('kPa:', np.round(pressure_kpa, 2))
print('°C:', np.round(temperature_c, 2))

In [ ]:
# Safe construction of a converted four-channel matrix.
converted = demo.copy()
converted[:, 0] = pressure_kpa
converted[:, 1] = temperature_c
print('converted channels: [kPa, °C, mm/s, bpd]')
print(np.round(converted, 2))

### Broadcasting illustration (dimensionless demonstration)

Suppose each column had a *reference scaling value* of the same unit as its column. Dividing the sensor matrix by these reference values creates dimensionless ratios. A vector of length 4 is broadcast across all six rows. **Warning:** choosing arbitrary reference values has no physical justification; these are teaching values only.

In [ ]:
reference = np.array([3500, 200, 6, 400], dtype=float)
ratios = demo / reference
print('shape:', ratios.shape)
print('first row normalized ratios:', np.round(ratios, 3))

# D — Comparison masks, missing measurements and logical conditions (25 minutes)

### Concept introduction

A **boolean mask** is an array of True/False values created from a condition, for example `pressure > 3500`. You can use it to select only qualifying readings or count alarms with `np.count_nonzero(mask)`. Combine conditions with **`&` (and)** and **`|` (or)**, using parentheses around each comparison.

**Missing values** require special attention. In a numeric ndarray, `np.nan` indicates a missing measurement. `np.isnan` detects it. `np.mean` returns NaN when missing values are present; `np.nanmean` ignores them (unless every value is missing). A zero can be a valid sensor reading, a sensor fault or a sentinel value—do not assume it means missing without documentation.

**Engineering example:** flag observations above an illustrative high-pressure threshold of 3500 psi or a vibration threshold of 6.0 mm/s. These are **synthetic training thresholds, not field operating limits**.

**Common mistakes:** writing `arr > 3500 and arr < 4000` instead of `(arr > 3500) & (arr < 4000)`; confusing `np.nan` with zero; calculating a threshold on already-converted kPa using a value intended for psi.

**Predict:** How many high-pressure events are in the demonstration array?

In [ ]:
high_pressure = demo[:, 0] > 3500
high_vibration = demo[:, 2] > 6.0
both = high_pressure & high_vibration
print('high pressure flags:', high_pressure)
print('high pressure count:', np.count_nonzero(high_pressure))
print('both flags:', both)
print('both rows:', demo[both])

In [ ]:
print('missing cells:', np.count_nonzero(np.isnan(demo)))
print('missing flow:', np.isnan(demo[:, 3]))
print('flow ordinary mean:', np.mean(demo[:, 3]))
print('flow nanmean:', np.nanmean(demo[:, 3]))

### D: Practice checkpoint

Make a mask for vibration at least 5.0 mm/s, count matching observations, and show pressure values for those rows. Confirm that the condition uses **mm/s**, not an unrelated unit.

In [ ]:
vibration_at_least_five = demo[:, 2] >= 5.0
print('count:', np.count_nonzero(vibration_at_least_five))
print('corresponding pressure psi:', demo[vibration_at_least_five, 0])

# E — Aggregation by sensor column and interpreting results (25 minutes)

### Concept introduction

**Aggregation** reduces many measurements to a summary: mean, minimum, maximum, sum or count. With a 2D array, **`axis=0`** performs the operation down the rows, resulting in **one value for each channel**. **`axis=1`** performs it across columns, resulting in one value per observation.

**Why the axis matters:** `np.nanmean(demo, axis=0)` returns four results in channel order. Those results must each retain their own unit. Averaging all four columns together would mix psi, °F, mm/s and bpd, producing a number with no physical meaning.

### Correct approaches

- `np.nanmean(demo, axis=0)` summaries by channel.
- `np.nanmax(demo[:, 0])` highest recorded pressure.
- `np.count_nonzero(~np.isnan(demo), axis=0)` nonmissing count per channel.

**Common mistakes:** forgetting `axis=0`; ignoring missing values, thereby returning NaN when you expected a numeric mean; labeling the returned vector as if it were a single scalar.

**Predict:** How many elements are in the result of `np.nanmean(demo, axis=0)`?

In [ ]:
channel_means = np.nanmean(demo, axis=0)
channel_counts = np.count_nonzero(~np.isnan(demo), axis=0)
for name, mean_value, n in zip(channel_names, channel_means, channel_counts):
    print(f'{name}: mean={mean_value:.2f}; available readings={n}')

In [ ]:
print('maximum pressure (psi):', np.nanmax(demo[:, 0]))
print('minimum pressure (psi):', np.nanmin(demo[:, 0]))
print('mean pressure (psi):', round(np.nanmean(demo[:, 0]), 2))

# F — Wrap-up and transition to independent lab (20 minutes)

### What you learned

A numeric sensor matrix is a compact representation of multi-channel readings. Your array's **shape and dtype** tell you what you have; **indexing** tells you what you are selecting; **vectorization** lets you apply engineering formulas across readings; **masks** isolate conditions; **NaN-aware aggregation** produces useful summaries without inventing values.

### Independent lab (assessed separately)

Open `Week10_Class1_Student_Lab_Guide.docx` and the *Student_Practical_Files* folder. Use the lab CSV rather than the demonstration array. Implement the solution as `class_01_numpy/numpy_sensor_lab.py` in your own repository. Your submission must include a README, requirements and Pull Request. The Jupyter Notebook is **not** the graded deliverable.

### Exit ticket (answer aloud, no code copying)

1. What does an ndarray shape of `(12, 4)` mean?
ans= It means the array has:
- 12 rows (observations or records).
- 4 columns (variables or measurements).
2. How do you isolate a pressure column?
ans= pressure = demo[:, 0]
3. Why is `np.nanmean` safer than `np.mean` for a column containing missing readings?
ans= np.mean() may return NaN if the column contains a missing value.
np.nanmean() calculates the average using the available numbers.
4. What is the difference between a Python list and an ndarray when using `* 2`?
ans= Python list: Repeats the list two times.
NumPy ndarray: Multiplies every element by 2.
5. Why should pressure and temperature never be aggregated into one average?
Pressure is measured in PSI.
Temperature is measured in °F.

**Next session:** Pandas Series and DataFrames — labels, tabular structures and basic selection.